# PySpark Gym — 03: Joins

Practice: inner / left / anti / semi joins, multi-table joins, broadcast hints, and self-joins.
Each problem builds a result DataFrame; assign it to the named `solution_N` variable and run the check cell.

In [1]:
from pathlib import Path
import sys

_cwd = Path.cwd()
_candidates = [
    _cwd / "pyspark",
    _cwd,
    _cwd.parent,
    _cwd.parent / "pyspark",
    _cwd.parent.parent,
    _cwd.parent.parent / "pyspark",
]
_pyspark_dir = next((p for p in _candidates if (p / "utils" / "__init__.py").exists()), None)
if _pyspark_dir is None:
    raise RuntimeError(
        "Cannot locate pyspark/utils. Run: uv run jupyter lab from the project root."
    )

if str(_pyspark_dir) not in sys.path:
    sys.path.insert(0, str(_pyspark_dir))

DATA_DIR = _pyspark_dir / "data"

from utils import get_spark, check
import pyspark.sql.functions as F
from pyspark.sql import Window

spark = get_spark()
spark.sparkContext.setLogLevel("ERROR")

customers = spark.read.csv(str(DATA_DIR / "customers.csv"), header=True, inferSchema=True)
products = spark.read.csv(str(DATA_DIR / "products.csv"), header=True, inferSchema=True)
orders = spark.read.csv(str(DATA_DIR / "orders.csv"), header=True, inferSchema=True)
order_items = spark.read.csv(str(DATA_DIR / "order_items.csv"), header=True, inferSchema=True)

for df in [customers, products, orders, order_items]:
    df.cache()

print(f"customers:   {customers.count():>6,}")
print(f"products:    {products.count():>6,}")
print(f"orders:      {orders.count():>6,}")
print(f"order_items: {order_items.count():>6,}")
from utils.checks.joins import Checker

checker = Checker(spark, customers, products, orders, order_items)
from utils.checks.joins import Checker

checker = Checker(spark, customers, products, orders, order_items)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/04 11:09:47 WARN Utils: Your hostname, dinmCND2520NNR, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/04 11:09:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/04 11:09:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


customers:      500
products:       100
orders:       8,000
order_items: 20,101


## Problem 1: Enriched Order Line Items

Build a fully-enriched line-item table by joining all four source tables, then filter to completed orders only.

<details>
<summary>Hint</summary>

Join `order_items` → `orders` (on `order_id`) → `products` (on `product_id`) → `customers` (on `customer_id`).
Alias `products.name` to `product_name` and `customers.name` to `customer_name` before or during the join to avoid
ambiguity. Compute `line_total` as `round(quantity * unit_price, 2)`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| order_id | int | |
| order_date | date | |
| customer_name | string | from `customers.name` |
| tier | string | bronze / silver / gold / platinum |
| product_name | string | from `products.name` |
| category | string | |
| quantity | int | |
| unit_price | double | |
| line_total | double | `round(quantity * unit_price, 2)` |

Expected: all completed-order line items, unordered.

In [ ]:
products_r = products.withColumnRenamed("name", "product_name")
customers_r = customers.withColumnRenamed("name", "customer_name")

solution_1 = (
    order_items.join(orders, "order_id")
    .filter(orders.status == "completed")
    .join(products_r, "product_id")
    .join(customers_r, "customer_id")
    .select(
        orders.order_id,
        orders.order_date,
        customers_r.customer_name,
        customers_r.tier,
        products_r.product_name,
        products_r.category,
        order_items.quantity,
        order_items.unit_price,
        F.round(order_items.quantity * order_items.unit_price, 2).alias("line_total"),
    )
)

solution_1.show()

+--------+----------+------------------+------+--------------------+---------------+--------+----------+----------+
|order_id|order_date|     customer_name|  tier|        product_name|       category|quantity|unit_price|line_total|
+--------+----------+------------------+------+--------------------+---------------+--------+----------+----------+
|       2|2024-06-21|    Charles Wilson|  gold|PeakGear Sports I...|         Sports|       2|    190.56|    381.12|
|       4|2023-07-13| Christopher Brown|bronze|PureTaste Food & ...|Food & Beverage|       4|     33.47|    133.88|
|       4|2023-07-13| Christopher Brown|bronze|DeskPro Office Su...|Office Supplies|       2|      94.9|     189.8|
|       5|2024-07-08|     Charles Smith|bronze|UrbanThread Cloth...|       Clothing|       4|    195.48|    781.92|
|       5|2024-07-08|     Charles Smith|bronze|NestCraft Home & ...|  Home & Garden|       4|    470.66|   1882.64|
|       5|2024-07-08|     Charles Smith|bronze| FunZone Toys Item 4|    

In [17]:
checker.p1(solution_1)

True

## Problem 2: Customers Who Have Never Ordered

Find customers with no record in the orders table — a classic anti-join pattern.

<details>
<summary>Hint</summary>

Build the set of `customer_id` values that appear in `orders` (use `.select("customer_id").distinct()`),
then do a **left anti join** from `customers` against that set. Anti join keeps only left-side rows that
have *no* matching key on the right.

</details>

| Column | Type | Notes |
|--------|------|-------|
| customer_id | int | sorted ASC |
| name | string | |
| email | string | |
| tier | string | |

Expected: all customers absent from `orders`, sorted by `customer_id` ASC.

In [29]:
customer_ordered = orders.select("customer_id").distinct()

solution_2 = (
    customers.join(customer_ordered, on="customer_id", how="left_anti")
    .select("customer_id", "name", "email", "tier")
    .orderBy(F.asc("customer_id"))
)

solution_2.show()

+-----------+----+-----+----+
|customer_id|name|email|tier|
+-----------+----+-----+----+
+-----------+----+-----+----+



In [30]:
checker.p2(solution_2)

True

## Problem 3: Most Frequently Bought Product Pairs

Find the top 10 pairs of products that appear together most often in the same order.

<details>
<summary>Hint</summary>

Self-join `order_items` as aliases `"a"` and `"b"` on `order_id`, filter
`a.product_id < b.product_id` (avoids duplicate pairs and self-pairs), then count.
Use `.alias("a")` / `.alias("b")` and reference columns with `F.col("a.product_id")`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| product_id_1 | int | the smaller product_id in the pair |
| product_id_2 | int | the larger product_id in the pair |
| times_bought_together | long | sorted DESC, top 10 |

Expected: 10 rows, ordered by `times_bought_together` DESC.

In [ ]:
solution_3 = (
    order_items.alias("a")
    .join(order_items.alias("b"), on="order_id")
    .filter(F.col("a.product_id") < F.col("b.product_id"))
    .groupBy("a.product_id", "b.product_id")
    .agg(F.count(F.col("order_id")).alias("times_bought_together"))
    .select(
        F.col("a.product_id").alias("product_id_1"),
        F.col("b.product_id").alias("product_id_2"),
        "times_bought_together",
    )
    .orderBy(F.desc("times_bought_together"))
    .limit(10)
)

solution_3.show()

+------------+------------+---------------------+
|product_id_1|product_id_2|times_bought_together|
+------------+------------+---------------------+
|          38|          94|                   13|
|          58|          98|                   13|
|          24|          71|                   13|
|           4|          40|                   13|
|          81|          89|                   13|
|           9|          70|                   12|
|           4|          23|                   12|
|          63|          95|                   12|
|           2|          33|                   12|
|           1|          62|                   11|
+------------+------------+---------------------+



In [43]:
checker.p3(solution_3)

True

## Problem 4: Customers Who Bought From Both Electronics AND Sports

Find customers who have purchased at least one item from the **Electronics** category *and*
at least one item from the **Sports** category.

<details>
<summary>Hint</summary>

1. Join `order_items` with `products` to get category per line item.
2. Join with `orders` to get `customer_id` per line item.
3. Filter to Electronics; take distinct `customer_id` → call it `elec_customers`.
4. Filter to Sports; take distinct `customer_id` → call it `sport_customers`.
5. Inner join the two sets on `customer_id` — only IDs present in *both* survive.
6. Join back to `customers` to get `name`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| customer_id | int | sorted ASC |
| name | string | |

Expected: customers present in both category buyer sets, sorted by `customer_id` ASC.

In [ ]:
order_items_product = order_items.join(products, "product_id")

electronics_customers = (
    order_items_product.join(orders, "order_id")
    .filter(products.category == "Electronics")
    .select("customer_id")
    .distinct()
)
sports_customers = (
    order_items_product.join(orders, "order_id")
    .filter(products.category == "Sports")
    .select("customer_id")
    .distinct()
)

solution_4 = (
    electronics_customers.join(sports_customers, "customer_id", how="inner")
    .join(customers, "customer_id")
    .select("customer_id", customers.name)
    .orderBy(F.asc("customer_id"))
)

solution_4.show()

+-----------+-----------------+
|customer_id|             name|
+-----------+-----------------+
|          1|Christopher Brown|
|          2|      Linda Jones|
|          3|     John Jackson|
|          4|    Michael Davis|
|          5| Michael Thompson|
|          6|   Linda Anderson|
|          7|    Daniel Wilson|
|          9|Elizabeth Ramirez|
|         10|   Nancy Anderson|
|         11|   Sarah Martinez|
|         12|   Charles Miller|
|         13| Matthew Martinez|
|         14| William Anderson|
|         15| Elizabeth Miller|
|         16|        Karen Lee|
|         17|Jennifer Anderson|
|         18|      Linda Perez|
|         21|      Michael Lee|
|         22| Robert Rodriguez|
|         24|    Richard Lopez|
+-----------+-----------------+
only showing top 20 rows


In [60]:
checker.p4(solution_4)

True

## Problem 5: Revenue Share by Customer Tier

Show how much each customer tier contributes to total revenue as a percentage.

<details>
<summary>Hint</summary>

Join `orders` with `customers` on `customer_id`, group by `tier`, sum
`total_amount` → `tier_revenue`. Collect the grand total with
`orders.agg(F.sum("total_amount")).first()[0]` and divide each tier's revenue by it
to get `revenue_share_pct`.

</details>

| Column | Type | Notes |
|--------|------|-------|
| tier | string | sorted ASC |
| tier_revenue | double | `round(sum(total_amount), 2)` |
| revenue_share_pct | double | `round(tier_revenue / grand_total * 100, 2)` |

Expected: one row per tier, sorted by `tier` ASC.

In [65]:
solution_5 = (
    orders.join(customers, "customer_id")
    .groupBy(customers.tier)
    .agg(F.round(F.sum("total_amount"), 2).alias("tier_revenue"))
    .withColumn(
        "revenue_share_pct",
        F.round(F.col("tier_revenue") / orders.agg(F.sum("total_amount")).first()[0] * 100, 2),
    )
    .orderBy(F.asc("tier"))
)

solution_5.show()

+--------+------------+-----------------+
|    tier|tier_revenue|revenue_share_pct|
+--------+------------+-----------------+
|  bronze|  9328972.47|            59.04|
|    gold|   2432036.9|            15.39|
|platinum|   991541.03|             6.28|
|  silver|  3048792.12|            19.29|
+--------+------------+-----------------+



In [66]:
checker.p5(solution_5)

True